# BBDM: Planck 100 GHz -> ACT+Planck 90 GHzСуперразрешение карт реликтового излучения диффузионной моделью наброуновском мосте.**Что важно помнить при работе с этим ноутбуком:**- Направление моста: `t = 0 -> y` (ACT+Planck, цель), `t = T -> x0` (Planck, вход).  В коде `x0` означает "патч Planck", а не "состояние модели в t=0".- В сэмплер попадает **только** Planck. Целевая карта на инференсе не  передаётся никуда и ни в каком виде.- Любая спектральная метрика считается **по >= 30-60 патчам**. Одиночный  патч на высоких ell шумит так, что систематика и шум оценки визуально  неотличимы.- Изменение `BBDM.loss` / `BBDM.q_sample` требует полного переобучения.  Изменение `BBDM.sample` / `_posterior_coeffs` -- нет, достаточно  пересэмплировать существующий чекпоинт (так делаются абляции по ETA и S).

### 0. Окружение (Colab)

In [ ]:
# Colab: подключить Drive и достать репозиторий.
# Локально этот блок не нужен.

# from google.colab import drive
# drive.mount("/content/drive")
# !pip install -q astropy scikit-image tqdm
# !git clone https://github.com/Perf0rator4/bbdm-cmb.git /content/bbdm_cmb

print("Skipped — running from a local checkout")

### 1. Импорты, пути, устройство

In [ ]:
import os
import sys

# Корень репозитория -- папка, в которой лежит пакет bbdm.
for _candidate in (os.getcwd(), os.path.abspath(".."),
                   "/content/bbdm_cmb", "/content/bbdm-cmb"):
    if os.path.isdir(os.path.join(_candidate, "bbdm", "model")):
        REPO_ROOT = _candidate
        break
else:
    raise RuntimeError("Пакет bbdm не найден — задайте REPO_ROOT вручную")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt

from bbdm.config import (
    PATCH_SIZE, TRAIN_RATIO, VAL_RATIO, SEED,
    MAX_ZERO_FRAC, MAX_MASK_MISMATCH_FRAC,
    IN_CH, BASE_CH, TIME_DIM, GROUPS,
    T, S, S_VAR, ETA,
    N_EPOCHS, BATCH_SIZE, LR, NUM_WORKERS, SPECTRAL_WEIGHT,
    EMA_START, EMA_DECAY,
)
from bbdm.data import CMBPatchDataset, compute_normalization, get_tile_splits
from bbdm.model import BBDM, UNet
from bbdm.train import train
from bbdm.sample import run_inference, visualize_inference
from bbdm.evaluate import (
    evaluate_image_metrics,
    evaluate_spectra,
    plot_spectral_comparison,
    print_band_table,
)

# Пути к данным. Либо пропишите их здесь, либо в bbdm/config.py --
# все функции ниже принимают их явными аргументами.
PLANCK_DIR     = "data/Diffusion/Planck/f100/"
ACT_DIR        = "data/Diffusion/Planck+ACT/f090/"
CHECKPOINT_DIR = "checkpoint/"

# На A40 (40 ГБ) помещается 32; на меньшей карте уменьшите.
BATCH_SIZE = 4

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Repo root: {REPO_ROOT}")
print(f"Device:    {device}")
if device == "cuda":
    print(f"GPU:       {torch.cuda.get_device_name(0)}")
    print(f"VRAM:      {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"\nETA = {ETA}  (0 — как в статье BBDM: x_T = вход ровно)")
print(f"SPECTRAL_WEIGHT = {SPECTRAL_WEIGHT}")

### 2. Разбиение train/val/testРазбиение делается **на уровне тайлов**, до нарезки на патчи: четыре патчаодного тайла соседние по небу, и разбиение на уровне патчей протащило быпочти одинаковое небо из train в test.

In [ ]:
train_tiles, val_tiles, test_tiles = get_tile_splits(
    planck_dir=PLANCK_DIR,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    seed=SEED,
)

### 3. НормализацияОбщие (mu, sigma) по ненулевым пикселям **только train-тайлов**, одни и теже для Planck и ACT: предсказание и цель денормализуются одной парой, и вTransfer Function нормировка сокращается.

In [ ]:
mu, sigma = compute_normalization(
    planck_dir=PLANCK_DIR,
    act_dir=ACT_DIR,
    train_tiles=train_tiles,
    patch_size=PATCH_SIZE,
)
print(f"mu    = {mu:.4f} uK")
print(f"sigma = {sigma:.4f} uK")

### 4. ДатасетыПатч выбрасывается, если у него слишком много замаскированных пикселей(`MAX_ZERO_FRAC`) **или** если маски Planck и ACT заметно не совпадают(`MAX_MASK_MISMATCH_FRAC`) -- у инструментов разные футпринты, и на частипатчей у одного из них срезан угол. Смотрите на печать "dropped ... byPlanck/ACT mask mismatch": если отсеивается заметная доля данных, порогстоит поднять осознанно, а не молча.

In [ ]:
common = dict(
    planck_dir=PLANCK_DIR,
    act_dir=ACT_DIR,
    mu=mu, sigma=sigma,
    patch_size=PATCH_SIZE,
    max_zero_frac=MAX_ZERO_FRAC,
    max_mask_mismatch_frac=MAX_MASK_MISMATCH_FRAC,
)

train_ds = CMBPatchDataset(tile_list=train_tiles, augment=True,  **common)
val_ds   = CMBPatchDataset(tile_list=val_tiles,   augment=False, **common)
test_ds  = CMBPatchDataset(tile_list=test_tiles,  augment=False, **common)

print(f"\nTrain: {len(train_ds)} samples ({len(train_ds.pairs)} unique pairs)")
print(f"Val:   {len(val_ds)} samples")
print(f"Test:  {len(test_ds)} samples")

x0, y = train_ds[0]
print(f"Patch shape: x0={tuple(x0.shape)}, y={tuple(y.shape)}")

### 5. Sanity check: как выглядят пары

In [ ]:
n_aug = len(train_ds.augmentations)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for col in range(4):
    # Шаг n_aug — чтобы взять 4 РАЗНЫХ патча, а не один в 4 аугментациях.
    x0, y = train_ds[col * n_aug]
    x0_np = train_ds.denormalize(x0[0].numpy())
    y_np  = train_ds.denormalize(y[0].numpy())

    # Общая шкала по цели: при индивидуальной автошкале разница в
    # амплитуде между входом и целью визуально исчезает.
    valid = y_np[y_np != 0]
    vmin, vmax = np.percentile(valid, [2, 98])

    for row, (data, label) in enumerate([(x0_np, "Planck"), (y_np, "ACT+Planck")]):
        ax = axes[row, col]
        im = ax.imshow(data, cmap="RdBu_r", vmin=vmin, vmax=vmax, origin="lower")
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(label, fontsize=12)

plt.suptitle("Dataset sanity check (общая шкала в каждой колонке)", fontsize=13)
plt.tight_layout()
plt.show()

### 6. Модель`spectral_weight` -- вес члена `L1(log RAPSD)` в лоссе. Чистый MSE тянетпредсказание к условному среднему, которое глаже любой отдельной истиннойреализации, и на высоких ell мощность недобирается (TF ~ 0.73 в прогоне 3при ETA=0). Спектральный член штрафует это отклонение напрямую, так чтоTF ~ 1 становится следствием того, на что модель обучена, а не случайнойкомпенсации двух противоположных дефектов.

In [ ]:
unet = UNet(in_ch=IN_CH, base_ch=BASE_CH, time_dim=TIME_DIM, groups=GROUPS)

bbdm = BBDM(
    model=unet,
    T=T,
    s=S_VAR,
    eta=ETA,                          # 0.0
    spectral_weight=SPECTRAL_WEIGHT,  # 0.1
)

n_params = sum(p.numel() for p in unet.parameters())
print(f"UNet parameters: {n_params:,}")
print(f"UNet size:       {n_params * 4 / 1024**2:.1f} MB")

### 7. Sanity check: один батч

In [ ]:
bbdm = bbdm.to(device)

x0_test, y_test = train_ds[0]
x0_test = x0_test.unsqueeze(0).to(device)
y_test  = y_test.unsqueeze(0).to(device)

with torch.no_grad():
    total, terms = bbdm.loss(x0_test, y_test, return_terms=True)

print(f"loss  = {total.item():.6f}")
print(f"  mse  = {terms['mse'].item():.6f}")
print(f"  spec = {terms['spec'].item():.6f}")
if device == "cuda":
    print(f"VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print("Forward pass OK")

### 8. Обучение`resume=True` подхватит `last.pt`, если сессия оборвалась. Валидационныйлосс считается с фиксированным сидом, поэтому val-кривая сравнима междуэпохами -- иначе и выбор лучшего чекпоинта, и `ReduceLROnPlateau`управлялись бы шумом оценки.

In [ ]:
bbdm, ema = train(
    bbdm=bbdm,
    train_dataset=train_ds,
    val_dataset=val_ds,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    ema_start=EMA_START,
    spectral_weight=SPECTRAL_WEIGHT,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    num_workers=NUM_WORKERS,
    resume=False,
)

### 9. Загрузка лучшего чекпоинтаЖивыми весами становятся EMA-веса: сначала грузится полный `state_dict`модели, затем веса сети перезаписываются EMA-тенью.

In [ ]:
checkpoint = torch.load(
    os.path.join(CHECKPOINT_DIR, "best.pt"),
    map_location=device,
    weights_only=False,
)
bbdm.load_state_dict(checkpoint["model"])
bbdm.model.load_state_dict(checkpoint["ema"])
bbdm = bbdm.to(device).eval()

print(f"Epoch:    {checkpoint['epoch'] + 1}")
print(f"Val loss: {checkpoint['val_loss']:.6f} "
      f"(mse {checkpoint['val_mse']:.6f}, spec {checkpoint['val_spec']:.4f})")
print(f"Hparams:  {checkpoint['hparams']}")

assert checkpoint["hparams"]["T"] == bbdm.T, "T чекпоинта не совпадает с моделью"
assert checkpoint["hparams"]["s"] == bbdm.s, "s чекпоинта не совпадает с моделью"

### 10. Инференс: три стохастических реализацииТри сэмпла из одного и того же Planck-патча. Они обязаны различаться:задача принципиально один-ко-многим, и одинаковые сэмплы означали бы, чтомодель схлопнулась в условное среднее.

In [ ]:
x0_val, y_val = val_ds[0]
x0_np = val_ds.denormalize(x0_val[0].numpy())
y_np  = val_ds.denormalize(y_val[0].numpy())

samples = run_inference(
    bbdm=bbdm,
    x0_norm=x0_val.numpy(),   # только Planck; цель сюда не передаётся
    mu=mu, sigma=sigma,
    S=S,
    device=device,
    n_samples=3,
    seed=0,
    progress=True,
)

visualize_inference(x0_np, y_np, samples)

spread = np.std([s for s in samples], axis=0).mean()
print(f"Средний разброс между сэмплами: {spread:.3f} uK "
      f"(0 означало бы схлопывание в условное среднее)")

### 11. Спектральная оценка: Transfer Function и r_ellОбе величины усредняются по 50 патчам, по схеме "среднее спектров, потомотношение". Среднее поштучных отношений здесь не годится: если у одногопатча мощность цели в каком-то бине близка к нулю, его отношение уноситсреднее.TF ~ 1 сам по себе ничего не доказывает. Смотреть надо вместе с `r_ell`:если лишняя мощность на высоких ell не коррелирует с целью (белый шум,галлюцинированные точечные источники), TF может быть близка к 1 припроседающей `r_ell` -- это не физическая точность, а совпадение.

In [ ]:
N_EVAL = 50
eval_indices = list(range(min(N_EVAL, len(val_ds))))

res = evaluate_spectra(
    bbdm=bbdm,
    dataset=val_ds,
    mu=mu, sigma=sigma,
    indices=eval_indices,
    S=S,
    device=device,
    batch_size=4,
)

print_band_table(res, "BBDM, ETA=0 + spectral loss")
plot_spectral_comparison([res], ["ETA=0 + spectral loss"])

### 12. Абляция по ETA (без переобучения)`ETA` -- дисперсия шума в стартовом состоянии обратного процесса. Она невходит в лосс, поэтому её можно менять и пересэмплировать **тот же самый**чекпоинт: `evaluate_spectra(eta=...)` подменяет её временно и возвращаетобратно.Что здесь проверяется (диагноз из прогона 3): шум `sqrt(ETA) * randn`спектрально **белый**. На низких ell реальный сигнал его перекрывает, а навысоких, где сигнал слаб, он примерно удваивал мощность -- отсюда TF ~ 2при ETA=0.01. При ETA=0 та же добавка исчезает, и становится виденнедобор мощности от MSE.Решающая проверка -- `r_ell`: у ETA=0 она должна быть **выше** на высокихell, несмотря на TF дальше от 1, потому что лишняя мощность при ETA=0.01не коррелирует с целью.

In [ ]:
ablation = {}
for eta_value in (0.0, 0.01):
    ablation[eta_value] = evaluate_spectra(
        bbdm=bbdm,
        dataset=val_ds,
        mu=mu, sigma=sigma,
        indices=eval_indices,   # те же патчи -- иначе сравниваются разные куски неба
        S=S,
        device=device,
        batch_size=4,
        eta=eta_value,
        seed=0,                 # и тот же шум
    )
    print_band_table(ablation[eta_value], f"ETA={eta_value}")

plot_spectral_comparison(
    [ablation[0.0], ablation[0.01]],
    ["ETA = 0", "ETA = 0.01"],
    title="Влияние стартового шума на TF и r_ell",
)
print(f"\nbbdm.eta после абляции: {bbdm.eta} (должно вернуться к исходному)")

### 12b. Откуда берётся избыток мощности на высоких ellДва измерения, отвечающие на разные вопросы.**`bridge_snr`** — модели не требует, считается по одному датасету. Мостподмешивает **белый** шум с пиксельной дисперсией `delta_t = 2s·m_t(1-m_t)`,а сигнальная компонента в `x_t` равна `(1-m_t)·y`. Спектр падает на ~6порядков от низких ell к высоким, поэтому плоский шумовой пол хоронитвысокие ell почти при любом t. Колонка `usable t` — доля обучающих шагов,на которых полоса вообще различима. Там, где она мала, сеть физически неможет научиться этому масштабу: это количественное обоснование §7.2.**`diagnose_prediction_spectrum`** — что реально делает обученная сеть за**один** шаг. Апостериорное среднее на слабой моде равно `k_t·x_t` с`k_t ≈ p/m_t ≪ 1`, то есть сеть обязана давить свой вход в десятки раз.Если вместо этого `actual` много больше `ideal` и почти не убывает сростом `t` — сеть пропускает белый шум моста на выход. Тогда `TF`выходит примерно на отношение «шум моста / мощность цели», а `r_ell`падает в ноль, потому что этот избыток по построению не связан с целью.

In [ ]:
from bbdm.evaluate import (
    bridge_snr, print_bridge_snr,
    diagnose_prediction_spectrum, print_prediction_spectrum,
)

snr = bridge_snr(bbdm, val_ds, n_patches=32, batch_size=8, device=device)
print_bridge_snr(snr)

In [ ]:
diag = diagnose_prediction_spectrum(
    bbdm, val_ds,
    t_values=(1, 10, 50, 100, 250, 500, 750, 999),
    n_patches=16, batch_size=4, device=device,
)
print_prediction_spectrum(diag)

### 13. PSNR / SSIMПриводятся для сопоставимости с литературой. SSIM для стохастическойгенерации один-ко-многим -- слабая метрика: она штрафует любую реализацию,отличную от конкретной наблюдённой, даже при идеальной статистике.Основной критерий здесь -- TF и r_ell выше.

In [ ]:
metrics = evaluate_image_metrics(
    bbdm=bbdm,
    dataset=val_ds,
    mu=mu, sigma=sigma,
    indices=eval_indices,
    S=S,
    device=device,
    batch_size=4,
)

print(f"PSNR: {metrics['psnr_mean']:.2f} ± {metrics['psnr_std']:.2f} dB")
print(f"SSIM: {metrics['ssim_mean']:.4f} ± {metrics['ssim_std']:.4f}")
print(f"(по {metrics['n_patches']} патчам)")

### 14. Тестовый сплит: визуальная проверка

In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.choice(len(test_ds), size=min(3, len(test_ds)), replace=False)

for i in indices:
    x0_t, y_t = test_ds[int(i)]
    x0_np = test_ds.denormalize(x0_t[0].numpy())
    y_np  = test_ds.denormalize(y_t[0].numpy())

    samples = run_inference(
        bbdm=bbdm,
        x0_norm=x0_t.numpy(),
        mu=mu, sigma=sigma,
        S=S, device=device, n_samples=3, seed=int(i),
    )
    visualize_inference(x0_np, y_np, samples)

### 15. Спектральная оценка на тестеФинальная цифра для статьи. Валидационный сплит использовался при выборечекпоинта и подборе `spectral_weight`, поэтому итоговые TF и r_ell надоприводить по тесту.

In [ ]:
res_test = evaluate_spectra(
    bbdm=bbdm,
    dataset=test_ds,
    mu=mu, sigma=sigma,
    n_patches=min(60, len(test_ds)),
    S=S,
    device=device,
    batch_size=4,
)

print_band_table(res_test, "TEST")
plot_spectral_comparison([res_test], ["test"], title="Test split")